# F1 Winner Prediction - Notebook 00: Setup & Data Collection

Clona el repo desde GitHub, instala dependencias, descarga datos de FastF1,
crea base de datos de circuitos y obtiene clima historico.

In [ ]:
# @title 1. Clone Repo & Install
!git clone https://github.com/USERNAME/f1_transformer.git
%cd f1_transformer
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['COLAB'] = '1'

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import torch
import pandas as pd
import numpy as np
print(f'PyTorch {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'No GPU')

In [ ]:
# @title 2. Fetch Race Data (FastF1 API)
# This runs the data collection script from src/
# It fetches all F1 race results 2014-2025 with rate limit handling

from src.data_collection.fetch_fastf1 import main
# Note: This will take 30-60 min for all 12 seasons due to API rate limits
# If data already exists in Drive, this step can be skipped

print('Running fetch_fastf1...')
print('This fetches data from FastF1 API with proper rate limiting.')
print('If you already have race_results_all.csv, skip this cell.')

# Uncomment to fetch:
# main()  # Fetches all 2014-2025 seasons

In [ ]:
# @title 3. Create Circuit Database
import pandas as pd
from pathlib import Path

# Static curated circuit data
circuits_data = [
    ('Albert Park', 5.278, 16, 4, 5, 2, 2, 2, 2, 79.8),
    ('Bahrain', 5.412, 15, 3, 5, 3, 1, 3, 2, 90.9),
    ('Baku', 6.003, 20, 2, -28, 1, 1, 2, 1, 100.3),
    ('Barcelona', 4.675, 16, 2, 100, 3, 2, 2, 1, 76.0),
    ('Hungaroring', 4.381, 14, 2, 150, 3, 3, 2, 1, 76.2),
    ('Interlagos', 4.309, 15, 2, 780, 3, 2, 2, 2, 70.5),
    ('Imola', 4.909, 19, 2, 50, 3, 2, 2, 1, 75.4),
    ('Jeddah', 6.174, 27, 3, 5, 1, 1, 1, 3, 87.4),
    ('Las Vegas', 6.201, 17, 2, 600, 1, 1, 2, 2, 94.0),
    ('Losail', 5.38, 16, 2, 5, 3, 2, 3, 2, 83.6),
    ('Marina Bay', 4.94, 23, 3, 5, 1, 3, 2, 1, 95.2),
    ('Mexico City', 4.304, 17, 3, 2240, 3, 2, 1, 2, 77.3),
    ('Miami', 5.412, 19, 3, 2, 2, 2, 2, 2, 88.5),
    ('Monaco', 3.337, 19, 2, 5, 1, 3, 2, 1, 70.4),
    ('Monza', 5.793, 11, 2, 160, 3, 1, 2, 3, 79.3),
    ('Montreal', 4.361, 14, 3, 5, 2, 1, 2, 2, 73.0),
    ('Mugello', 5.245, 15, 2, 250, 3, 2, 2, 1, 78.5),
    ('Paul Ricard', 5.842, 15, 2, 400, 3, 2, 2, 2, 88.5),
    ('Portimao', 4.653, 15, 2, 100, 3, 2, 2, 2, 77.4),
    ('Red Bull Ring', 4.318, 10, 3, 700, 3, 1, 2, 2, 65.0),
    ('Sepang', 5.543, 15, 2, 50, 3, 2, 3, 2, 94.2),
    ('Shanghai', 5.451, 16, 2, 5, 3, 2, 3, 2, 92.0),
    ('Silverstone', 5.891, 18, 3, 150, 3, 2, 2, 3, 87.1),
    ('Sochi', 5.848, 18, 2, 5, 2, 2, 2, 2, 90.8),
    ('Spa', 7.004, 19, 2, 450, 3, 1, 2, 3, 101.9),
    ('Suzuka', 5.807, 18, 2, 100, 3, 3, 2, 2, 89.2),
    ('Yas Marina', 5.281, 21, 2, 5, 3, 2, 2, 2, 85.2),
    ('Zandvoort', 4.259, 14, 2, 5, 3, 3, 2, 1, 71.3),
    ('Austin', 5.513, 20, 2, 170, 3, 2, 2, 2, 96.0),
    ('Hockenheim', 4.574, 17, 2, 100, 3, 2, 2, 2, 73.7),
    ('Nurburgring', 5.148, 15, 2, 600, 3, 2, 2, 2, 87.1),
    ('Istanbul', 5.338, 14, 2, 57, 3, 2, 2, 2, 84.7),
    ('Monte Carlo', 3.337, 19, 2, 5, 1, 3, 2, 1, 70.4),
]
cols = ['circuit_name','length_km','corners','drs_zones','altitude_m',
        'track_type_code','downforce_code','tyre_degradation_code','overtaking_code','lap_record_s']

RAW = Path('/content/drive/MyDrive/f1_transformer/data/raw')
(RAW / 'circuits').mkdir(parents=True, exist_ok=True)
circuits_df = pd.DataFrame([dict(zip(cols, c)) for c in circuits_data])
circuits_df.to_csv(RAW / 'circuits' / 'circuits.csv', index=False)
print(f'Saved {len(circuits_df)} circuits')

In [ ]:
# @title 4. Fetch Weather Data (Open-Meteo)
import requests
from tqdm.notebook import tqdm

# Load race data to get unique races
races_df = pd.read_csv(RAW / 'races' / 'race_results_all.csv')
races_df['event_date'] = pd.to_datetime(races_df['event_date'], format='mixed')
unique_races = races_df[['year','round','circuit','event_date']].drop_duplicates()

# Coordinates for each circuit
coords = {
    'Albert Park': (-37.85,144.97), 'Bahrain': (26.03,50.51),
    'Baku': (40.37,49.85), 'Barcelona': (41.57,2.26),
    'Hungaroring': (47.58,19.25), 'Interlagos': (-23.70,-46.70),
    'Imola': (44.34,11.71), 'Jeddah': (21.63,39.10),
    'Las Vegas': (36.11,-115.17), 'Losail': (25.49,51.45),
    'Marina Bay': (1.29,103.86), 'Mexico City': (19.40,-99.09),
    'Miami': (25.96,-80.24), 'Monaco': (43.73,7.42),
    'Monza': (45.62,9.28), 'Montreal': (45.50,-73.52),
    'Mugello': (43.99,11.37), 'Nurburgring': (50.33,6.95),
    'Paul Ricard': (43.25,5.79), 'Portimao': (37.23,-8.64),
    'Red Bull Ring': (47.22,14.76), 'Sepang': (2.76,101.74),
    'Shanghai': (31.34,121.22), 'Silverstone': (52.07,-1.02),
    'Sochi': (43.41,39.97), 'Spa': (50.44,5.97),
    'Suzuka': (34.84,136.54), 'Yas Marina': (24.47,54.60),
    'Zandvoort': (52.39,4.54), 'Austin': (30.13,-97.64),
    'Hockenheim': (49.33,8.57), 'Istanbul': (40.95,29.40),
    'Monte Carlo': (43.73,7.42),
}

weather_records = []
BASE_URL = 'https://archive-api.open-meteo.com/v1/archive'

for _, race in tqdm(unique_races.iterrows(), total=len(unique_races)):
    circuit = str(race['circuit'])
    coord = None
    for name, c in coords.items():
        if name.lower() in circuit.lower() or circuit.lower() in name.lower():
            coord = c; break
    if coord is None: continue
    date = race['event_date']
    if pd.isna(date): continue
    params = {
        'latitude': coord[0], 'longitude': coord[1],
        'start_date': date.strftime('%Y-%m-%d'), 'end_date': date.strftime('%Y-%m-%d'),
        'daily': 'temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_mean,surface_pressure_mean',
        'timezone': 'auto'
    }
    try:
        resp = requests.get(BASE_URL, params=params, timeout=10)
        data = resp.json()
        if 'daily' in data and data['daily']['temperature_2m_mean']:
            d = data['daily']
            weather_records.append({
                'year': int(race['year']), 'round': int(race['round']),
                'circuit': circuit, 'event_date': date,
                'openmeteo_temp_mean': d['temperature_2m_mean'][0],
                'openmeteo_humidity': d['relative_humidity_2m_mean'][0],
                'openmeteo_precip_mm': d['precipitation_sum'][0],
                'openmeteo_wind_speed': d['wind_speed_10m_mean'][0],
                'openmeteo_pressure': d['surface_pressure_mean'][0],
                'openmeteo_rain_flag': 1 if d['precipitation_sum'][0] > 0.5 else 0,
            })
        import time; time.sleep(0.15)
    except: continue

weather_df = pd.DataFrame(weather_records)
weather_df.to_csv(RAW / 'weather' / 'weather_all.csv', index=False)
print(f'Saved {len(weather_df)} weather records')

In [ ]:
# @title 5. Validate Data
df = pd.read_csv(RAW / 'races' / 'race_results_all.csv')
print('='*60)
print('DATA VALIDATION')
print('='*60)
print(f'Total entries: {len(df)}')
print(f'Seasons: {sorted(df["year"].unique())}')
print(f'Drivers: {df["driver_abbreviation"].nunique()}')
print(f'Constructors: {df["constructor"].nunique()}')
print(f'Circuits: {df["circuit"].nunique()}')
print(f'\nYear breakdown:')
for y in sorted(df['year'].unique()):
    yr = df[df['year'] == y]
    w = yr[yr['finish_position'] == 1]['driver_abbreviation'].values[:3]
    print(f'  {y}: {yr["round"].nunique()} races, top winners: {list(w)}')
print(f'\nHas quali data: {df["q1_time"].notna().sum()}/{len(df)}')
print(f'DNF rate: {df["dnf"].mean()*100:.1f}%')
print('\nReady for Notebook 01!')